# Data input pipeline for the SNN baseline — CIFAR-10 / CIFAR-100

This notebook walks through how images get from disk into the spiking network.
It uses the **same code the training CLI uses** (`neuron_baseline.data.build_loaders`
and `neuron_baseline.model.build_net_for`), so what you see here is exactly what
training sees.

```
raw CIFAR image (32x32 RGB, uint8)
   │
   ├─ train split ─► RandomCrop(32, pad=4) ─► RandomHorizontalFlip ─┐
   │                                                                 ├─► ToTensor ─► Normalize(mean, std per channel)
   └─ val / test ──────────────────────────────────────────────────┘
   │
   ▼
seeded split: 45,000 train / 5,000 val (from the training set) · 10,000 test
   │
   ▼
DataLoader  [B, 3, 32, 32]   (train: shuffle, drop_last · eval: no drop_last, bigger batch)
   │
   ▼
direct encoding: the SAME normalized image is fed at each of the T = 50 time steps
   │
   ▼
Conv(3→12,5) → Pool → LIF → Conv(12→64,5) → Pool → LIF → Linear(1600→10/100) → LIF
   │
   ▼
output spike counts over T → argmax = prediction
```

Set `DATASET` below to `"CIFAR10"` or `"CIFAR100"` and run all cells.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
import numpy as np
import matplotlib.pyplot as plt

from neuron_baseline.config import Config
from neuron_baseline.data import DATASETS, build_loaders, build_transforms, dataset_info, canonical_name
from neuron_baseline.model import build_net_for, forward_pass

DATASET = "CIFAR10"          # or "CIFAR100"
DATA_DIR = "../data"         # same folder the CLI uses (snn-surrogate-baseline/data)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

cfg = Config(dataset=canonical_name(DATASET), data_dir=DATA_DIR, device=str(device))
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print("dataset:", cfg.dataset, "| info:", dataset_info(cfg.dataset))

## 1. Build the loaders

Same call as training. The split is seeded (`seed = 20260923`, the repo default),
so every run sees the same 5,000 validation images.

In [ ]:
loaders = build_loaders(
    cfg.data_dir, cfg.batch_size, cfg.eval_batch_size, cfg.val_size, cfg.seed,
    num_workers=0, pin_memory=(device.type == "cuda"),
    dataset=cfg.dataset, augment=cfg.augment,
)
for split, dl in loaders.items():
    print(f"{split:5s}: {len(dl.dataset):6d} images, {len(dl):4d} batches "
          f"(batch_size={dl.batch_size}, drop_last={dl.drop_last})")

xb, yb = next(iter(loaders["train"]))
print("\nbatch images:", tuple(xb.shape), xb.dtype, "| labels:", tuple(yb.shape), yb.dtype)
print("pixel range after normalization: [%.2f, %.2f]" % (xb.min(), xb.max()))

train_tf, eval_tf = build_transforms(cfg.dataset, cfg.augment)
print("\ntrain transform:\n", train_tf)
print("eval transform:\n", eval_tf)

## 2. Check the normalization

After `Normalize(mean, std)` each colour channel of the (clean) validation images
should have mean ≈ 0 and std ≈ 1.

In [ ]:
s = torch.zeros(3); s2 = torch.zeros(3); n = 0
for x, _ in loaders["val"]:
    s += x.sum(dim=(0, 2, 3)); s2 += (x ** 2).sum(dim=(0, 2, 3)); n += x.numel() // 3
mean = s / n; std = (s2 / n - mean ** 2).sqrt()
spec = DATASETS[cfg.dataset]
print("normalization constants  mean:", spec["mean"], " std:", spec["std"])
print("val set after normalize  mean:", [round(v, 4) for v in mean.tolist()],
      " std:", [round(v, 4) for v in std.tolist()])

## 3. Class balance in each split

In [ ]:
def split_labels(ds):
    if hasattr(ds, "indices"):                     # Subset (train / val)
        return np.array(ds.dataset.targets)[ds.indices]
    return np.array(ds.targets)                     # full test set

num_classes = dataset_info(cfg.dataset)["num_classes"]
counts = {k: np.bincount(split_labels(loaders[k].dataset), minlength=num_classes)
          for k in ("train", "val", "test")}
for k, c in counts.items():
    print(f"{k:5s}: per-class min={c.min():5d} max={c.max():5d} total={c.sum()}")

fig, ax = plt.subplots(figsize=(10, 3))
w = 0.28
for i, k in enumerate(("train", "val", "test")):
    ax.bar(np.arange(num_classes) + (i - 1) * w, counts[k] / counts[k].sum(), w, label=k)
ax.set_xlabel("class id"); ax.set_ylabel("fraction of split"); ax.set_title(f"{cfg.dataset} class balance")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Sample images (un-normalized for display)

In [ ]:
classes = loaders["test"].dataset.classes
m = torch.tensor(spec["mean"]).view(3, 1, 1); sd = torch.tensor(spec["std"]).view(3, 1, 1)
show = lambda t: (t * sd + m).clamp(0, 1).permute(1, 2, 0).numpy()

xv, yv = next(iter(loaders["val"]))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lab in zip(axes.flat, xv[:16], yv[:16]):
    ax.imshow(show(img)); ax.set_title(classes[lab], fontsize=9); ax.axis("off")
plt.suptitle(f"{cfg.dataset} validation samples"); plt.tight_layout(); plt.show()

## 5. Training-time augmentation

The same training image passed through the train transform 8 times. Each epoch the
network sees a slightly shifted / flipped copy, which reduces overfitting.
Validation and test images are never augmented.

In [ ]:
raw_ds = loaders["train"].dataset.dataset          # underlying CIFAR dataset (train transform)
idx = loaders["train"].dataset.indices[0]
img_pil, lab = raw_ds.data[idx], raw_ds.targets[idx]
from PIL import Image
img_pil = Image.fromarray(img_pil)

fig, axes = plt.subplots(1, 9, figsize=(15, 2.2))
axes[0].imshow(show(eval_tf(img_pil))); axes[0].set_title("original"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(show(train_tf(img_pil))); ax.set_title("augmented"); ax.axis("off")
plt.suptitle(f"class: {classes[lab]}"); plt.tight_layout(); plt.show()

## 6. How the image enters the spiking network (direct encoding)

The SNN runs for `T = 50` time steps. At **every** step it receives the same
normalized image (no Poisson / rate encoding). The first conv layer turns pixels
into input current; the LIF neurons integrate that current over time and fire
spikes when their membrane potential crosses the threshold.

Below: an **untrained** network (random weights), just to show the shape of the
signal at each stage. Spike rates change once the network is trained.

In [ ]:
torch.manual_seed(cfg.seed)
net = build_net_for(cfg).to(device).eval()
x1 = xv[:1].to(device)

taps = {}
hooks = [net[i].register_forward_hook(lambda m, i_, o, k=k: taps.setdefault(k, []).append(o[0] if isinstance(o, tuple) else o))
         for k, i in (("LIF 1", 2), ("LIF 2", 5), ("Output LIF", 8))]
with torch.no_grad():
    spk_rec, mem_rec, _ = forward_pass(net, cfg.num_steps, x1)
for h in hooks: h.remove()

print("input fed at each step:", tuple(x1.shape), " x T =", cfg.num_steps)
for k, v in taps.items():
    v = torch.stack(v)                               # [T, 1, ...]
    print(f"{k:10s}: per-step shape {tuple(v.shape[1:])}, mean firing rate {v.float().mean().item():.3f} spikes/neuron/step")
print("output spike record:", tuple(spk_rec.shape), "-> prediction = argmax of spike count over T")

lif1 = torch.stack(taps["LIF 1"])[:, 0].flatten(1).cpu()      # [T, neurons]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].imshow(show(xv[0])); axes[0].set_title(f"input ({classes[yv[0]]})"); axes[0].axis("off")
sel = lif1[:, :200].T
axes[1].imshow(sel, aspect="auto", cmap="Greys", interpolation="nearest")
axes[1].set_xlabel("time step"); axes[1].set_ylabel("neuron"); axes[1].set_title("LIF 1 spike raster (200 neurons)")
axes[2].plot(mem_rec[:, 0].cpu().numpy()); axes[2].axhline(cfg.threshold, ls="--", c="k", lw=0.8)
axes[2].set_xlabel("time step"); axes[2].set_ylabel("membrane potential"); axes[2].set_title("output neurons")
plt.tight_layout(); plt.show()

## 7. Loader throughput and GPU forward time

In [ ]:
t0 = time.perf_counter(); n = 0
for i, (x, y) in enumerate(loaders["train"]):
    n += x.shape[0]
    if i == 49: break
dt = time.perf_counter() - t0
print(f"train loader (num_workers=0): {n / dt:,.0f} images/s")

xg = xb.to(device)
with torch.no_grad():
    forward_pass(net, cfg.num_steps, xg)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    forward_pass(net, cfg.num_steps, xg)
    if device.type == "cuda": torch.cuda.synchronize()
print(f"SNN forward, batch {xg.shape[0]}, T={cfg.num_steps} on {device}: {(time.perf_counter() - t0) * 1000:.0f} ms")

## 8. Summary

| Item | Value |
|---|---|
| Image size | 32 × 32 × 3 (RGB) |
| Split | 45,000 train / 5,000 val / 10,000 test (seed 20260923) |
| Normalization | per-channel mean / std (training-set statistics) |
| Augmentation | RandomCrop(32, padding=4) + RandomHorizontalFlip, train only |
| Batch | 128 train, 256 eval |
| Input encoding | direct: same image at each of T = 50 steps |
| Network | Tutorial 6 conv SNN, 3 input channels, 10 or 100 output LIF neurons |

## 9. Train on the GPU

From `snn-surrogate-baseline/`:

```bash
python -m neuron_baseline smoke
python -m neuron_baseline train --dataset CIFAR10  --epochs 20 --num-workers 4
python -m neuron_baseline train --dataset CIFAR100 --epochs 20 --num-workers 4
```

Results go to `runs/cifar10-<timestamp>/` (config, checkpoints, results.json, plots).